Real-World Use Case — Healthcare Chatbot

A healthcare chatbot that:

Blocks off-topic or harmful requests

Redacts patient PII (emails, credit card numbers)

Requires human approval before booking appointments

Validates that outputs are medically appropriate

In [1]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.messages import AIMessage

# --- Healthcare-specific content filter ---
class HealthcareSafetyFilter(AgentMiddleware):
    """Block non-medical or harmful requests in a healthcare context."""

    BLOCKED_TOPICS = ["drug synthesis", "self-harm", "suicide method", "weapon", "hack"]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_msg = state["messages"][0]
        if first_msg.type != "human":
            return None

        content = first_msg.content.lower()
        for topic in self.BLOCKED_TOPICS:
            if topic in content:
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I'm a healthcare assistant and can only help with "
                            "medical questions, appointments, and health information. "
                            "If you're in crisis, please call 112 or your local emergency number."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


# --- Medical output validator ---
class MedicalOutputValidator(AgentMiddleware):
    """Ensure all responses include appropriate medical disclaimers."""

    DISCLAIMER = "\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*"

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Add disclaimer if not already present
        if "medical advice" not in last_message.content.lower():
            last_message.content += self.DISCLAIMER

        return None
# --- Healthcare tools ---
@tool
def search_symptoms(symptoms: str) -> str:
    """Search for information about medical symptoms."""
    return f"Symptom information for: {symptoms}. Please consult a doctor for diagnosis."

@tool
def book_appointment(patient_name: str, date: str, doctor: str) -> str:
    """Book a medical appointment."""
    return f"Appointment booked for {patient_name} with Dr. {doctor} on {date}"

@tool
def get_medication_info(medication: str) -> str:
    """Get information about a medication."""
    return f"General info about {medication}. Always follow your doctor's prescription."


# --- Build the healthcare chatbot ---
healthcare_bot = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_symptoms, book_appointment, get_medication_info],
    middleware=[
        # Guardrail 1: Block harmful/off-topic requests
        HealthcareSafetyFilter(),

        # Guardrail 2: Redact patient PII from inputs
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Guardrail 3: Require approval before booking appointments
        HumanInTheLoopMiddleware(
            interrupt_on={
                "book_appointment": True,
                "search_symptoms": False,
                "get_medication_info": False,
            }
        ),

        # Guardrail 4: Add medical disclaimer to all outputs
        MedicalOutputValidator(),
    ],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a helpful healthcare assistant. "
        "You can search for symptoms, medication information, and help book appointments. "
        "Always be empathetic and remind users to consult a doctor for diagnosis."
    )
)

print("🏥 Healthcare chatbot with full guardrail stack created!")

🏥 Healthcare chatbot with full guardrail stack created!


In [2]:
# Test 1: Safe medical query
config_t1 = {"configurable": {"thread_id": "healthcare_session_t1"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "What are symptoms of Type 2 Diabetes?"}]},
    config=config_t1
)

result


{'messages': [HumanMessage(content='What are symptoms of Type 2 Diabetes?', additional_kwargs={}, response_metadata={}, id='d9b7a648-f936-4c38-a274-bb7c2c0a2df4'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'mcznxfg37', 'function': {'arguments': '{"symptoms":"Type 2 Diabetes"}', 'name': 'search_symptoms'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 460, 'total_tokens': 491, 'completion_time': 0.087481599, 'completion_tokens_details': None, 'prompt_time': 0.036818672, 'prompt_tokens_details': None, 'queue_time': 0.048511657, 'total_time': 0.124300271}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a110f8-a328-7b22-85af-b7737141b21f-0', tool_calls=[{'name': 'search_symptoms', 'args': {'symptoms': 'Type 2 Diabetes'}, 'id': 'mcznxfg37', 'type': 'tool_call'}], invalid_t

In [3]:
# Test 2: Query with PII (email gets redacted)
result = healthcare_bot.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is patient123@gmail.com. What can I take for a headache?"
    }]},
    config=config_t1
)
print("=== PII Redaction Test ===")
print(result["messages"][-1].content)

=== PII Redaction Test ===
I’m sorry you’re dealing with a headache—that can be really uncomfortable.

For most occasional, mild-to-moderate headaches, over-the-counter (OTC) pain relievers are commonly used. Here are some general options:

**Common OTC Medications:**
*   **Acetaminophen (Tylenol):** Often a first choice, especially for tension-type headaches. It’s generally easier on the stomach.
*   **Ibuprofen (Advil, Motrin) or Naproxen (Aleve):** These are NSAIDs (non-steroidal anti-inflammatory drugs) that can help with pain and inflammation.
*   **Aspirin:** Another NSAID option, though it’s less commonly recommended as a first-line treatment for simple headaches compared to acetaminophen or ibuprofen unless directed by a doctor.

**Important Considerations:**
*   **Follow label instructions:** Do not exceed the recommended dose.
*   **Stomach issues:** If you have a history of stomach ulcers, bleeding problems, or kidney issues, avoid NSAIDs (ibuprofen, naproxen, aspirin) unles

In [4]:
# Test 3: Off-topic / harmful request — gets blocked
result = healthcare_bot.invoke({
    "messages": [{"role": "user", "content": "How do I synthesize drugs at home?"}]
},
 config=config_t1)
print("=== Blocked Request ===")
print(result["messages"][-1].content)

=== Blocked Request ===
I cannot provide instructions for synthesizing drugs or any illegal substances. Creating controlled substances or other illicit drugs at home is dangerous, illegal, and poses serious risks to your health and safety.

If you or someone you know is struggling with substance use, I encourage you to reach out for support. Here are some helpful resources:

*   **SAMHSA National Helpline:** 1-800-662-HELP (4357) – Free, confidential, 24/7 treatment referral and information service (in English and Spanish) for individuals and family members facing mental and/or substance use disorders.
*   **988 Suicide & Crisis Lifeline:** Call or text 988 for immediate support if you are in crisis.
*   **Local healthcare provider:** A doctor can provide guidance on safe, legal, and effective ways to manage pain or other medical conditions.

Please prioritize your safety and well-being by seeking professional help through legal and appropriate channels.

⚕️ *This is general health inf

In [5]:
# Test 4: Appointment booking — requires human approval
config = {"configurable": {"thread_id": "healthcare_session_001"}}

result = healthcare_bot.invoke(
    {"messages": [{"role": "user", "content": "Book me an appointment with Dr. Sharma on March 15"}]},
    config=config
)
print("=== Appointment Booking — Awaiting Approval ===")
print(result)

# Approve
from langgraph.types import Command
approved = healthcare_bot.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)
print("\n=== After Approval ===")
print(approved["messages"][-1].content)

=== Appointment Booking — Awaiting Approval ===
{'messages': [HumanMessage(content='Book me an appointment with Dr. Sharma on March 15', additional_kwargs={}, response_metadata={}, id='6d55e268-8df0-4132-8130-6a0b7e00ddc7'), AIMessage(content="I'd be glad to help you book that appointment! However, I need your name to complete the booking. Could you please provide your full name?\n\n⚕️ *This is general health information, not medical advice. Please consult a qualified healthcare professional.*", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 464, 'total_tokens': 495, 'completion_time': 0.061331974, 'completion_tokens_details': None, 'prompt_time': 0.038485291, 'prompt_tokens_details': None, 'queue_time': 0.048147968, 'total_time': 0.099817265}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a110f9-